In [1]:
!uv add requests python-frontmatter

Resolved 125 packages in 11ms
Checked 120 packages in 9ms


In [86]:
import requests
import io
import zipfile
import frontmatter
from openai import OpenAI
from dotenv import load_dotenv

In [3]:
repo_owner='evidentlyai'
repo_name='docs'
branch_name='main'

zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'
zip_response=requests.get(zip_url)

In [4]:
zip_archive=zipfile.ZipFile(io.BytesIO(zip_response.content))

In [5]:
filenames=zip_archive.namelist()
filenames[20:30]

['docs-main/docs/library/prompt_optimization.mdx',
 'docs-main/docs/library/report.mdx',
 'docs-main/docs/library/synthetic_data_api.mdx',
 'docs-main/docs/library/tags_metadata.mdx',
 'docs-main/docs/library/tests.mdx',
 'docs-main/docs/platform/',
 'docs-main/docs/platform/alerts.mdx',
 'docs-main/docs/platform/dashboard_add_panels.mdx',
 'docs-main/docs/platform/dashboard_add_panels_ui.mdx',
 'docs-main/docs/platform/dashboard_overview.mdx']

In [6]:
filename='docs-main/docs/platform/alerts.mdx'
mdx_file = zip_archive.open(filename)
mdx_content =mdx_file.read().decode('utf-8')
print(mdx_content[:150])

---
title: 'Alerts'
description: 'How to set up alerts.'
---

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **


In [7]:
post=frontmatter.loads(mdx_content)
print(post.content[:100])
print(post.metadata)
_, filename_corrected = filename.split("/",maxsplit=1)
print(filename_corrected)

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently En
{'title': 'Alerts', 'description': 'How to set up alerts.'}
docs/platform/alerts.mdx


In [8]:
docs={
    'content':post.content,
    'title':post.metadata.get('title'),
    'description':post.metadata.get('description'),
    'filename':filename_corrected
}

In [9]:
print(docs)

{'content': '<Check>\n  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n![](/images/alerts.png)\n\nTo enable alerts, open the Project and navigate to the "Alerts" in the left menu. You must set:\n\n* A notification channel.\n\n* An alert condition.\n\n## Notification channels\n\nYou can choose between the following options:\n\n* **Email**. Add email addresses to send alerts to.\n\n* **Slack**. Add a Slack webhook.\n\n* **Discord**. Add a Discord webhook.\n\n## Alert conditions\n\n### Failed tests\n\nIf you use Tests (conditional checks) in your Project, you can tie alerting to the failed Tests in a Test Suite. Toggle this option on the Alerts page. Evidently will set an alert to the defined channel if any of the Tests fail.\n\n<Tip>\n  **How to avoid alert fatigue?** Use the `is_critical` parameter to mark non-critical Test as Warnings. Setting it to `False` prevent alerts for those checks even if they fail.\n</Tip>\n\n

In [10]:
def read_github_repository(repo_owner,repo_name,branch="main"):
    url =f"https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch}.zip"
    response=requests.get(url)
    response.raise_for_status()

    documents=[]
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        for file_path in zip_ref.namelist():
            if not file_path.endswith(('.md','.mdx')):
                continue
            with zip_ref.open(file_path) as file:
                content=file.read().decode('utf-8')
                post = frontmatter.loads(content)
                doc={
                    'content':post.content,
                    'title':post.metadata.get('title'),
                    'descirption':post.metadata.get('description'),
                    'filename':file_path.split('/',1)[-1]
                }
                documents.append(doc)
    return documents

In [11]:
repo_owner = 'evidentlyai'
repo_name = 'docs'

documents = read_github_repository(repo_owner, repo_name)

print(f"Downloaded {len(documents)} documents")

Downloaded 95 documents


In [12]:
!uv add gitsource

Resolved 125 packages in 9ms
Checked 120 packages in 9ms


In [13]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")

Loaded 95 documents


In [14]:
document = files[10]

print(document.filename)
print(document.content[:160])


docs/library/output_formats.mdx
---
title: 'Output formats'
description: 'How to export the evaluation results.'
---

You can view or export Reports in multiple formats.

**Pre-requisites**:




In [15]:
import frontmatter 

post = frontmatter.loads(document.content)
data = post.to_dict()
data['filename'] = document.filename


In [16]:
document.parse()

{'title': 'Output formats',
 'description': 'How to export the evaluation results.',
 'content': 'You can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python

In [17]:
documents = [f.parse() for f in files]

In [18]:
print(len(documents))

95


In [19]:
documents[10]

{'title': 'Output formats',
 'description': 'How to export the evaluation results.',
 'content': 'You can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python

In [20]:
!uv add minsearch

Resolved 125 packages in 29ms
Checked 120 packages in 8ms


In [21]:
from minsearch import Index

In [22]:
# minsearch requires two  args 
# 1.text_fields -- fields to search with full-text search
# 2.keyword_fields -- fields for exact matching only

In [23]:
query = 'LLM as a Judge'

In [24]:
index=Index(
    text_fields =["title","description","content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [25]:
results = index.search(query, num_results=5)

In [26]:
len(results)

5

In [27]:
print(results[1])

{'title': 'LLM-as-a-jury', 'description': 'Evaluate the LLM outputs with multiple LLMs.', 'content': 'This evaluation approach uses multiple LLMs to evaluate the same output. You can do this to obtain an aggregate evaluation result — e.g., consider an output a "pass" only if all or the majority of LLMs approve — or to explicitly surface disagreements.\n\nBlog explaining the concept of LLM jury: https://www.evidentlyai.com/blog/llm-judges-jury .\n\nCode example as a Jupyter notebook: https://github.com/evidentlyai/community-examples/blob/main/tutorials/LLM_as_a_jury_Example.ipynb\n\n## Preparation\n\nInstall Evidently:\n\n```python\npip install evidently litellm \n```\n\n(Or install `evidently[llm]`.)\n\nImport the components you\'ll use:\n\n```python\nimport pandas as pd\nfrom evidently import Dataset\nfrom evidently import DataDefinition\nfrom evidently import Report\nfrom evidently.presets import TextEvals\nfrom evidently.tests import eq, is_in, not_in\nfrom evidently.descriptors imp

In [28]:
document_test =list(range(100))

In [29]:
document_test[:10]

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [32]:
window_size=10
start=0
while start < len(document_test):
    end = start+window_size
    chunk = document_test[start:end]
    print(chunk)
    start=end
    

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
[20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
[30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
[40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
[50, 51, 52, 53, 54, 55, 56, 57, 58, 59]
[60, 61, 62, 63, 64, 65, 66, 67, 68, 69]
[70, 71, 72, 73, 74, 75, 76, 77, 78, 79]
[80, 81, 82, 83, 84, 85, 86, 87, 88, 89]
[90, 91, 92, 93, 94, 95, 96, 97, 98, 99]


In [39]:
window_size=10
start=0
step=5
chunks=[]
while start<len(document_test):
    end =start+window_size
    chunk = document_test[start:end]
    chunks.append(chunk)
    if len(chunk) <window_size:
        break
    # print(chunk)
    start =start+step
print(chunks)


[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], [5, 6, 7, 8, 9, 10, 11, 12, 13, 14], [10, 11, 12, 13, 14, 15, 16, 17, 18, 19], [15, 16, 17, 18, 19, 20, 21, 22, 23, 24], [20, 21, 22, 23, 24, 25, 26, 27, 28, 29], [25, 26, 27, 28, 29, 30, 31, 32, 33, 34], [30, 31, 32, 33, 34, 35, 36, 37, 38, 39], [35, 36, 37, 38, 39, 40, 41, 42, 43, 44], [40, 41, 42, 43, 44, 45, 46, 47, 48, 49], [45, 46, 47, 48, 49, 50, 51, 52, 53, 54], [50, 51, 52, 53, 54, 55, 56, 57, 58, 59], [55, 56, 57, 58, 59, 60, 61, 62, 63, 64], [60, 61, 62, 63, 64, 65, 66, 67, 68, 69], [65, 66, 67, 68, 69, 70, 71, 72, 73, 74], [70, 71, 72, 73, 74, 75, 76, 77, 78, 79], [75, 76, 77, 78, 79, 80, 81, 82, 83, 84], [80, 81, 82, 83, 84, 85, 86, 87, 88, 89], [85, 86, 87, 88, 89, 90, 91, 92, 93, 94], [90, 91, 92, 93, 94, 95, 96, 97, 98, 99], [95, 96, 97, 98, 99]]


In [46]:
def sliding_window(text,size=1000,step=500):
    chunks=[]
    start=0
    text_length=len(text)
    
    while start < text_length:
        end= start+size
        chunk= text[start:end]
        chunks.append({'start':start,'content':chunk})
        start =end-step
        if end>=text_length:
            break

    return chunks

In [47]:
len(sliding_window(results[0]['content'],size=3000, step=2500))

39

In [50]:
documents[30]

{'title': 'Overview',
 'description': 'How production AI quality monitoring works.',
 'content': 'AI observability lets you evaluate the quality of the inputs and outputs of your AI application as it runs in production. This gives an up-to-date view of your system behavior and helps spot and fix issues.\n\nEvidently offers several ways to set up monitoring.\n\n## Batch monitoring jobs\n\n<Check>\n  Supported in: `Evidently OSS`, `Evidently Cloud` and `Evidently Enterprise`.\n</Check>\n\n**Best for**: batch ML pipelines, regression testing, and near real-time ML systems that don’t need instant quality evaluations.\n\n![](/images/monitoring_flow_batch.png)\n\n**How it works**:\n\n* **Build your evaluation pipeline**. Create a pipeline in your infrastructure to run monitoring jobs. This can be a Python script, cron job, or orchestrated with a tool like Airflow. Run it at regular intervals (e.g., hourly, daily) or trigger it when new data or labels arrive.\n\n* **Run metric calculations**.

In [62]:
document_chunks=[]

for doc in documents:
    if not doc.get('content'):
        continue
    copy=doc.copy()   

In [63]:
# del copy['content']
content =copy.pop('content')

In [64]:
copy

{'title': 'Why synthetic data?',
 'description': 'When do you need synthetic data in LLM evaluations.',
 'filename': 'synthetic-data/why_synthetic.mdx'}

In [65]:
content

"When working on an AI system, you need test data to run automated evaluations for quality and safety. A test dataset is a structured set of test cases. It can contain:\n\n* Just the inputs, or\n* Both inputs and expected outputs (ground truth).\n\nYou can use this test dataset to:\n\n* Run **experiments** and track if changes improve or degrade system performance.\n* Run **regression testing** to ensure updates don’t break what was already working.\n* **Stress-test** your system with complex or adversarial inputs to check its resilience.\n\n![](/images/synthetic/synthetic_experiments_img.png)\n\nYou can create test datasets manually, collect them from real or historical data, or generate them synthetically. While real data is best, it is not always available or sufficient to cover all cases. Public LLM benchmarks help with general model comparisons but don’t reflect your specific use case. Manually writing test cases takes time and effort.\n\n**Synthetic data helps here**. It’s especi

In [71]:
document_chunks=[]

for doc in documents:
    if not doc.get('content'):
        continue
    copy=doc.copy()
    content=copy.pop('content')
    chunks = sliding_window(content, size=3000, step=1500)
    for id, chunk in enumerate(chunks):
        chunk['chunk_id']=id
        chunk.update(copy)
        document_chunks.append(chunk)

In [72]:
document_chunks[5]

{'start': 1500,
 'content': 'manually?** See the section below for available options.\n</Note>\n\n**Step 4. Run evals.** Once the **Dataset** object is ready, you can [add Descriptors ](/docs/library/descriptors) and[ run Reports](/docs/library/report).\n\n### Special cases\n\n**Working directly with pandas.DataFrame**. You can sometimes pass a `pandas.DataFrame` directly to `report.run()` without creating the Dataset object. This works for checks like numerical/categorical data summaries or drift detection. However, it\'s best to always create a `Dataset` object explicitly for clarity and control.\n\n**Working with two datasets**. If you\'re working with current and reference datasets (e.g., for drift detection), create a Dataset object for each. Both must have identical data definition.\n\n## Data definition\n\nThis page shows all the different mapping options. Note that you **only need to use the relevant ones** that apply for your evaluation scenario. For example, you don’t need co

In [73]:
document_chunks[6]

{'start': 3000,
 'content': 'escriptors) without explicit mapping. But it\'s a good idea to map text columns since you may later run other evals which vary by column type.\n</Info>\n\n#### Tabular data\n\nMap numerical, categorical or datetime columns:\n\n```python\ndefinition = DataDefinition(\n    text_columns=["Latest_Review"],\n    numerical_columns=["Age", "Salary"],\n    categorical_columns=["Department"],\n    datetime_columns=["Joining_Date"]\n    )\n    \neval_data = Dataset.from_pandas(\n    source_df,\n    data_definition=definition\n)\n```\n\nExplicit mapping helps avoid mistakes like misclassifying numerical columns with few unique values as categorical.\n\n<Info>\n  If you **exclude** certain columns in mapping, they’ll be ignored in all evaluations.\n</Info>\n\n#### Default column types\n\nIf you do not pass explicit mapping, the following defaults apply:\n\n| **Column Type**       | **Description**                                                                         

In [75]:
chunk_index=Index(
    text_fields=["title","description","content"],
    keyword_fields=["filename"]
)
chunk_index.fit(document_chunks)

In [77]:
chunk_results=chunk_index.search(query)

In [78]:
len(chunk_results)

10

In [79]:
chunk_results[2]

{'start': 19500,
 'content': 'he message effectively.""",\n        target_category="concise",\n        non_target_category="verbose",\n        uncertainty="unknown",\n        include_reasoning=True,\n        pre_messages=[("system", "You are an expert text evaluator. You will be given a text of the response to a user question.")],\n        )\n```\n\nAdd this new descriptor to our existing dataset:\n\n```python\neval_dataset.add_descriptors(descriptors=[\n    LLMEval("new_response",\n            template=verbosity,\n            provider = "openai",\n            model = "gpt-4o-mini",\n            alias="Verbosity")\n    ])\n```\n\nRun the Report and view the summary results:\xa0\n\n```python\nreport = Report([\n    TextEvals()\n])\n\nmy_eval = report.run(eval_dataset, None)\nmy_eval\n```\n\n![](/images/examples/llm_judge_tutorial_verbosity-min.png)\n\nYou can also view the dataframe using `eval_dataset.as_dataframe()`\n\n<Info>\n  Don\'t fully agree with the results? Use these labels as

In [80]:
from gitsource import chunk_documents

In [82]:
document_chunks=chunk_documents(documents)
document_chunks[0]

{'start': 0,
 'content': '<Note>\n  If you\'re not looking to build API reference documentation, you can delete\n  this section by removing the api-reference folder.\n</Note>\n\n## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.\n\n<Card\n  title="Plant Store Endpoints"\n  icon="leaf"\n  href="https://github.com/mintlify/starter/blob/main/api-reference/openapi.json"\n>\n  View the OpenAPI specification file\n</Card>\n\n## Authentication\n\nAll API endpoints are authenticated using Bearer tokens and picked up from the specification file.\n\n```json\n"security": [\n  {\n    "bearerAuth": []\n  }\n]\n```',
 'title': 'Introduction',
 'description': 'Example section for showcasing API endpoints',
 'filename': 'api-reference/introduction.mdx'}

In [89]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(r"C:\Users\v2kri\OneDrive\ai\Alex\from-rag-to-agents\ai-engineer-6\.env")

True

In [90]:
openai_client = OpenAI()

In [91]:
search_result=chunk_index.search(query,num_results=5)

In [92]:
query = 'How do I implement llm as a judge?'

In [93]:
import json
search_result_json=json.dumps(search_result,indent=2)

In [95]:
instructions="""
You're a course assistant, your task is to answer the QUESTION from the course students using the provided CONTEXT
"""

user_prompt=f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

In [102]:
def llm(user_prompt, instructions=None,model='gpt-4o-mini'):
    messages=[]
    if instructions is not None:
        messages.append({
            "role":"system",
            "content":instructions
        })

    messages.append({
    "role":"user",
    "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages)
    
    return response.output_text



In [103]:
answer=llm(user_prompt, instructions)

In [104]:
print(answer)

To implement an LLM as a judge, follow these steps based on the tutorial provided:

1. **Set Up Your Environment**:
   - Ensure you have basic Python knowledge and access to an OpenAI API key.
   - Install the Evidently library:
     ```bash
     pip install evidently
     ```
   - Import necessary modules:
     ```python
     import pandas as pd
     import numpy as np
     from evidently import Dataset, Report
     from evidently.llm.templates import BinaryClassificationPromptTemplate
     import os
     os.environ["OPENAI_API_KEY"] = "YOUR_KEY"
     ```

2. **Create an Evaluation Dataset**:
   - Generate a toy dataset with:
     - Questions (inputs to the LLM)
     - Target responses (approved, accurate responses)
     - New responses (to be evaluated)
     - Manual labels indicating correctness (correct/incorrect).

   Here’s an example of how to set up your dataset:
   ```python
   data = [
       ["Hi there, how do I reset my password?", target_response, new_response, "incorrect"

In [108]:
def search(query):
    return chunk_index.search(query, num_results=5)

In [109]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
    <QUESTION>
    {query}
    </QUESTION>

    <CONTEXT>
    {search_result_json}
    </CONTEXT>
    """.strip()

    return user_prompt


In [113]:
def rag(query):
    search_results=search(query)
    prompt=build_prompt(query, search_results)
    answer = llm(prompt,instructions)
    return answer

In [114]:
rag('how do I implement llm as a judge?')

'To implement an LLM (Large Language Model) as a judge, follow these steps:\n\n1. **Installation and Setup**:\n   - Install the Evidently library:\n     ```python\n     pip install evidently\n     ```\n   - Import the necessary modules:\n     ```python\n     import pandas as pd\n     import numpy as np\n     from evidently import Dataset, DataDefinition, Report\n     from evidently.llm.templates import BinaryClassificationPromptTemplate\n     import os\n     os.environ["OPENAI_API_KEY"] = "YOUR_KEY"  # Set your OpenAI API key\n     ```\n\n2. **Create an Evaluation Dataset**:\n   - Generate a toy Q&A dataset that includes:\n     - **Questions**: Inputs sent to the LLM.\n     - **Target responses**: Approved responses deemed accurate.\n     - **New responses**: Responses to be evaluated.\n     - **Manual labels**: Indicate if the response is correct or incorrect.\n   - Example data structure:\n     ```python\n     data = [\n         ["Question", "Target Response", "New Response", "Correc